# MongoDB Checkpointer (NoSQL) — What's Actually Stored?

`MongoDBSaver` (custom, in `checkpointer.py`) writes LangGraph state to MongoDB documents.
Unlike SQLite/Postgres (rows in tables), MongoDB stores **JSON-like documents** in **collections**.

```
MongoDB database: langgraph
├── checkpoints          ← one document per super-step  (state snapshots)
└── checkpoint_writes    ← one document per pending write (fault tolerance)
```

This notebook shows the raw documents, explains every field, and compares
the NoSQL document structure to the SQL table structure.

> **Prerequisites:**  
> `docker compose up -d` in this folder (starts MongoDB on port 27017)  
> OR set `MONGO_URI` in your `.env` for MongoDB Atlas.

## 0 · Setup

In [ ]:
import os, sys
import pandas as pd
from IPython.display import display

_ENV = '/Users/datasense/Desktop/langgrapgh-agent/.env'
for line in open(_ENV):
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        k, v = line.split('=', 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from checkpointer import MongoDBSaver

import pymongo
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, MessagesState, START, END

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
DB_NAME   = os.getenv('MONGO_DB',  'langgraph')
print('Connecting to:', MONGO_URI, '/', DB_NAME)

## 1 · Connect and build the graph

In [ ]:
client = pymongo.MongoClient(MONGO_URI)
db     = client[DB_NAME]
saver  = MongoDBSaver(db)

llm = init_chat_model('openai:gpt-4o-mini', temperature=0)

def chat_node(state: MessagesState):
    return {'messages': [llm.invoke(state['messages'])]}

builder = StateGraph(MessagesState)
builder.add_node('chat', chat_node)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)

graph = builder.compile(checkpointer=saver)
print('Collections so far:', db.list_collection_names())
print('Graph compiled with MongoDBSaver')

## 2 · Run a 3-turn conversation

In [ ]:
alice = {'configurable': {'thread_id': 'mg-alice'}}
bob   = {'configurable': {'thread_id': 'mg-bob'}}

r1 = graph.invoke({'messages': [{'role': 'user', 'content': 'Hi! My name is Satvik and I love LangGraph.'}]}, alice)
print('Turn 1 (alice):', r1['messages'][-1].content[:70])

r2 = graph.invoke({'messages': [{'role': 'user', 'content': "What's my name and what do I love?"}]}, alice)
print('Turn 2 (alice):', r2['messages'][-1].content[:70])

r3 = graph.invoke({'messages': [{'role': 'user', 'content': 'What programming language do I probably use?'}]}, alice)
print('Turn 3 (alice):', r3['messages'][-1].content[:70])

r4 = graph.invoke({'messages': [{'role': 'user', 'content': "I'm Bob, totally new here."}]}, bob)
print('Turn 1 (bob):  ', r4['messages'][-1].content[:70])

print('\nCollections now:', db.list_collection_names())

---
## 3 · Inspect the raw MongoDB documents

### Collection: `checkpoints`

Each checkpoint is a **document** (Python dict / JSON object):

| Field | Type | What it stores |
|-------|------|----------------|
| `_id` | ObjectId | MongoDB auto-ID |
| `thread_id` | string | which conversation |
| `checkpoint_ns` | string | subgraph namespace (empty = root graph) |
| `checkpoint_id` | string | UUID for this snapshot |
| `parent_checkpoint_id` | string | previous snapshot UUID |
| `type` | string | serialisation format |
| `checkpoint` | Binary | serialised full state (all channel values) |
| `metadata` | Binary | serialised metadata object |
| `meta_step` | int | step number (indexed for queries) |
| `meta_source` | string | `loop` / `input` / `update` |

In [ ]:
docs = list(db.checkpoints.find(
    {}, {'_id':0, 'checkpoint':0, 'metadata':0}  # exclude blobs for display
).sort('checkpoint_id', pymongo.DESCENDING))

print(f'checkpoints collection: {len(docs)} documents\n')
for doc in docs:
    print(f"  thread={doc['thread_id']:<12}  "
          f"step={doc.get('meta_step','?'):>3}  "
          f"source={doc.get('meta_source','?'):<6}  "
          f"ckpt_id={doc['checkpoint_id'][:20]}...")
          
print()
print('One full document (without binary blobs):')
print(docs[0])

In [ ]:
rows = []
for doc in docs:
    rows.append({
        'thread_id'     : doc['thread_id'],
        'checkpoint_id' : doc['checkpoint_id'][:20] + '...',
        'parent_id'     : (doc.get('parent_checkpoint_id') or '-')[:16] + '...',
        'type'          : doc.get('type', '?'),
        'meta_step'     : doc.get('meta_step', '?'),
        'meta_source'   : doc.get('meta_source', '?'),
    })
display(pd.DataFrame(rows))

### Decoding a document — what is inside the binary fields?

In [ ]:
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
serde = JsonPlusSerializer()

# get the latest checkpoint for alice
doc = db.checkpoints.find_one(
    {'thread_id': 'mg-alice'},
    sort=[('checkpoint_id', pymongo.DESCENDING)]
)

print('Raw document fields:', list(doc.keys()))
print('checkpoint field size:', len(doc['checkpoint']), 'bytes')
print('metadata field size: ', len(doc['metadata']),   'bytes')
print()

state = serde.loads_typed((doc['type'], bytes(doc['checkpoint'])))
meta  = serde.loads_typed((doc['type'], bytes(doc['metadata'])))

print('Decoded metadata:')
print('  step   :', meta.get('step'))
print('  source :', meta.get('source'))
print('  writes :', list((meta.get('writes') or {}).keys()))

msgs = state.get('channel_values', {}).get('messages', [])
print(f'\nMessages stored in this checkpoint: {len(msgs)}')
for m in msgs:
    print(f'  [{m.type.upper():<6}] {m.content[:65]}')

### Collection: `checkpoint_writes`

Each pending node write is a **separate document**:

| Field | Type | What it stores |
|-------|------|----------------|
| `thread_id` | string | same as parent checkpoint |
| `checkpoint_id` | string | the in-progress checkpoint this belongs to |
| `task_id` | string | UUID of the node task |
| `task_path` | string | execution path (for nested graphs) |
| `idx` | int | write order within the task |
| `channel` | string | which state key was written (e.g. `messages`) |
| `type` | string | serialisation format |
| `value` | Binary | serialised value for that channel |

In [ ]:
write_docs = list(db.checkpoint_writes.find(
    {}, {'_id':0, 'value':0}
).sort('checkpoint_id', pymongo.DESCENDING))

print(f'checkpoint_writes collection: {len(write_docs)} documents')
rows = []
for d in write_docs:
    rows.append({
        'thread_id'     : d['thread_id'],
        'checkpoint_id' : d['checkpoint_id'][:16] + '...',
        'task_id'       : d['task_id'][:10] + '...',
        'idx'           : d['idx'],
        'channel'       : d['channel'],
        'type'          : d['type'],
    })
display(pd.DataFrame(rows))

---
## 4 · NoSQL vs SQL — key differences in storage

| | SQLite / Postgres (SQL) | MongoDB (NoSQL) |
|---|---|---|
| **Structure** | fixed rows in tables | flexible documents in collections |
| **Schema** | enforced (columns defined) | schema-free (any fields allowed) |
| **Binary data** | `BLOB` / `BYTEA` column | `Binary` field inside document |
| **Indexed metadata** | needs separate column | stored in same document (`meta_step`) |
| **Query style** | `SELECT * FROM checkpoints WHERE thread_id='alice'` | `db.checkpoints.find({'thread_id':'alice'})` |
| **Joins** | `JOIN checkpoint_writes ON checkpoint_id` | separate collection, queried independently |

The **LangGraph API is 100% identical** — `get_state`, `get_state_history`, `update_state`
behave exactly the same regardless of backend.

## 5 · LangGraph API view — same data, decoded

In [ ]:
snap = graph.get_state(alice)
msgs = snap.values['messages']
print('=== Latest state for mg-alice ===')
print(f'  step       : {snap.metadata["step"]}')
print(f'  checkpoint : {snap.config["configurable"]["checkpoint_id"][:24]}...')
print(f'  next nodes : {snap.next or "(done)"}')
print(f'  messages   : {len(msgs)}')
for i, m in enumerate(msgs, 1):
    print(f'    [{i}] {m.type.upper():<6} {m.content[:60]}')

In [ ]:
history = list(graph.get_state_history(alice))
rows = []
for snap in history:
    msgs = snap.values.get('messages', [])
    rows.append({
        'step'     : snap.metadata.get('step'),
        'ckpt_id'  : snap.config['configurable'].get('checkpoint_id', '')[:20] + '...',
        'msgs'     : len(msgs),
        'next'     : str(snap.next),
        'source'   : snap.metadata.get('source'),
    })
print(f'Full history for mg-alice: {len(history)} checkpoints')
display(pd.DataFrame(rows))

## 6 · All threads — MongoDB aggregation

In [ ]:
pipeline = [
    {'$group': {'_id': '$thread_id', 'checkpoints': {'$sum': 1},
                'latest': {'$max': '$checkpoint_id'}}},
    {'$sort':  {'latest': -1}}
]
results = list(db.checkpoints.aggregate(pipeline))
rows = [{'thread_id': r['_id'], 'checkpoints': r['checkpoints'],
         'latest_checkpoint': r['latest'][:20] + '...'} for r in results]
print(f'All threads in MongoDB ({len(rows)} found):')
display(pd.DataFrame(rows))

## Summary

```
MongoDB document layout
──────────────────────────────────────────────────────────────
 checkpoints collection      1 document per super-step per thread
    thread_id                which conversation
    checkpoint_id            UUID snapshot identifier
    checkpoint  (Binary)     full serialised graph state
    metadata    (Binary)     step, source, writes map
    meta_step   (int)        indexed copy of step for fast queries

 checkpoint_writes collection 1 document per channel write
    channel                  which state key was written
    value       (Binary)     the serialised channel value
    task_id                  which node task wrote this
──────────────────────────────────────────────────────────────
```

NoSQL means **flexible schema** — our custom MongoDBSaver also stores `meta_step` and
`meta_source` as top-level indexed fields so you can query checkpoints by step number
without deserialising the binary blob.

In [ ]:
client.close()
print('Connection closed.')